# 09 - E5 Baseline multiclase agrupado sagital

**Objetivo:** construir un primer baseline multiclase 2D sagital sobre SPIDER, partiendo del pipeline binario ya validado.

Este notebook evalua si una U-Net 2D simple puede diferenciar grupos tecnicos de etiquetas de la mascara, en lugar de segmentar solo foreground/background.

**Alcance:** modalidad T1, eje sagital 2, target `256x256`, mapping agrupado preliminar de 4 clases, seleccion de slice sin mascara con `center_window_best_prediction`.

**Fuera de alcance:** nnU-Net, 3D, axial, backend y nombres anatomicos definitivos sin documentacion formal del dataset.

## 1. Instalacion e importacion de dependencias

In [ ]:
!pip -q install SimpleITK scikit-image tqdm

In [ ]:
from pathlib import Path
import json
import random
import warnings

import SimpleITK as sitk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage.transform import resize
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("SimpleITK:", sitk.Version())

## 2. Verificacion de entorno y rutas

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA disponible:", torch.cuda.is_available())
print("Dispositivo:", DEVICE)
if DEVICE.type == "cpu":
    print("ADVERTENCIA: no se detecto GPU. El entrenamiento multiclase puede ser lento en CPU.")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
DATASET_ROOT = Path("/content/drive/MyDrive/PFI_MVP/data/SPIDER")
PREPROCESS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E4_preprocesamiento")
BINARY_IMPROVED_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E5_baseline_mejorado_binario")
HOLDOUT_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E5_holdout_validacion")
SLICE_EVAL_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E5_slice_selection_eval")
MULTICLASS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E5_multiclase_agrupado")
FIGURES_ROOT = Path("/content/drive/MyDrive/PFI_MVP/figures")
DOCS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/docs")

for path in [MULTICLASS_ROOT, FIGURES_ROOT, DOCS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

CANDIDATES_CSV = PREPROCESS_ROOT / "E4_baseline_candidates_no_space.csv"
HOLDOUT_SELECTED_CSV = HOLDOUT_ROOT / "E5_holdout_selected_cases.csv"
BINARY_MODEL_PATH = BINARY_IMPROVED_ROOT / "E5_improved_unet2d_binary_best.pt"
SLICE_SELECTION_REPORT_JSON = SLICE_EVAL_ROOT / "E5_slice_selection_report.json"

print("CANDIDATES_CSV:", CANDIDATES_CSV)
print("BINARY_MODEL_PATH:", BINARY_MODEL_PATH)
print("SLICE_SELECTION_REPORT_JSON:", SLICE_SELECTION_REPORT_JSON)
print("MULTICLASS_ROOT:", MULTICLASS_ROOT)

## 3. Carga de candidatos T1

In [ ]:
MODALITY_FILTER = "t1"
N_CASES = 100
SAGITTAL_AXIS = 2
TARGET_SIZE = (256, 256)
NUM_CLASSES = 4
BASE_CHANNELS = 16
BATCH_SIZE = 4
EPOCHS = 20
LEARNING_RATE = 1e-3
CENTER_WINDOW_RADIUS = 3
INFERENCE_BATCH_SIZE = 16

candidates_df = pd.read_csv(CANDIDATES_CSV)

def infer_case_modality(row):
    text = " ".join(str(v).lower() for v in row.values)
    if "t2" in text:
        return "t2"
    if "t1" in text:
        return "t1"
    return "unknown"


def ensure_case_id(df):
    out = df.copy()
    if "case_id" not in out.columns:
        for candidate in ["file_name", "image_path", "source_image_path"]:
            if candidate in out.columns:
                out["case_id"] = out[candidate].apply(lambda x: Path(str(x)).stem)
                break
    if "case_id" not in out.columns:
        raise ValueError("No se pudo construir case_id.")
    out["case_id"] = out["case_id"].astype(str)
    return out


candidates_df = ensure_case_id(candidates_df)
if "modality" not in candidates_df.columns:
    candidates_df["modality"] = candidates_df.apply(infer_case_modality, axis=1)

t1_df = candidates_df[candidates_df["modality"].str.lower().eq(MODALITY_FILTER)].copy()
if len(t1_df) == 0:
    raise RuntimeError("No hay candidatos T1 disponibles.")

selected_df = t1_df.sample(n=min(N_CASES, len(t1_df)), random_state=SEED).reset_index(drop=True)
selected_cases_csv_path = MULTICLASS_ROOT / "E5_multiclass_selected_cases.csv"
selected_df.to_csv(selected_cases_csv_path, index=False)

print("Candidatos totales:", len(candidates_df))
print("Candidatos T1:", len(t1_df))
print("Casos seleccionados:", len(selected_df))
print("CSV seleccion:", selected_cases_csv_path)
display(selected_df.head())

## 4. Analisis de labels originales

In [ ]:
def resolve_path(value):
    return Path(str(value))


def get_case_paths(row):
    image_candidates = ["image_path", "source_image_path", "image", "img_path"]
    mask_candidates = ["mask_path", "source_mask_path", "mask", "seg_path"]
    image_path = None
    mask_path = None
    for column in image_candidates:
        if column in row and pd.notna(row[column]):
            image_path = resolve_path(row[column])
            break
    for column in mask_candidates:
        if column in row and pd.notna(row[column]):
            mask_path = resolve_path(row[column])
            break
    if image_path is None or mask_path is None:
        raise ValueError("No se encontraron columnas de path para imagen/mascara.")
    return image_path, mask_path


def read_mha(path: Path):
    itk_image = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(itk_image)
    return itk_image, array


label_counts = {}
label_case_frequency = {}
total_voxels = 0

for _, row in tqdm(selected_df.iterrows(), total=len(selected_df), desc="Analizando labels"):
    _, mask_path = get_case_paths(row)
    _, mask = read_mha(mask_path)
    labels, counts = np.unique(mask, return_counts=True)
    total_voxels += int(mask.size)
    present_labels = set()
    for label, count in zip(labels, counts):
        label_int = int(label)
        label_counts[label_int] = label_counts.get(label_int, 0) + int(count)
        present_labels.add(label_int)
    for label_int in present_labels:
        label_case_frequency[label_int] = label_case_frequency.get(label_int, 0) + 1

label_rows = []
foreground_voxels = sum(count for label, count in label_counts.items() if label != 0)
for label in sorted(label_counts):
    label_rows.append({
        "original_label": int(label),
        "voxel_count": int(label_counts[label]),
        "case_frequency": int(label_case_frequency.get(label, 0)),
        "voxel_ratio_total": float(label_counts[label] / max(total_voxels, 1)),
        "foreground_ratio_within_foreground": float(label_counts[label] / max(foreground_voxels, 1)) if label != 0 else 0.0,
    })

original_label_distribution_df = pd.DataFrame(label_rows)
original_label_distribution_csv_path = MULTICLASS_ROOT / "E5_multiclass_original_label_distribution.csv"
original_label_distribution_df.to_csv(original_label_distribution_csv_path, index=False)

print("Labels originales globales:", original_label_distribution_df["original_label"].tolist())
print("CSV distribucion labels:", original_label_distribution_csv_path)
display(original_label_distribution_df)

## 5. Mapping multiclase agrupado

El mapping siguiente es tecnico y preliminar. No asigna nombres anatomicos definitivos porque este notebook no incorpora documentacion formal del dataset que relacione labels con estructuras.

In [ ]:
LABEL_GROUP_MAPPING = {
    "0": {
        "name": "background",
        "original_labels": [0],
    },
    "1": {
        "name": "technical_group_low_labels_1_9",
        "original_labels": list(range(1, 10)),
    },
    "2": {
        "name": "technical_group_label_100",
        "original_labels": [100],
    },
    "3": {
        "name": "technical_group_high_labels_201_208",
        "original_labels": list(range(201, 209)),
    },
}

original_to_group = {}
for group_id, config in LABEL_GROUP_MAPPING.items():
    for original_label in config["original_labels"]:
        original_to_group[int(original_label)] = int(group_id)

mapping_json_path = MULTICLASS_ROOT / "E5_multiclass_label_mapping.json"
mapping_json_path.write_text(json.dumps(LABEL_GROUP_MAPPING, indent=2), encoding="utf-8")

print("Mapping exportado:", mapping_json_path)
print(json.dumps(LABEL_GROUP_MAPPING, indent=2))

## 6. Modelos y funciones de preprocesamiento

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class SimpleUNet2D(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, base_channels=16):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, base_channels)
        self.enc2 = DoubleConv(base_channels, base_channels * 2)
        self.enc3 = DoubleConv(base_channels * 2, base_channels * 4)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base_channels * 4, base_channels * 8)
        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, 2, stride=2)
        self.dec3 = DoubleConv(base_channels * 8, base_channels * 4)
        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 2, stride=2)
        self.dec2 = DoubleConv(base_channels * 4, base_channels * 2)
        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, 2, stride=2)
        self.dec1 = DoubleConv(base_channels * 2, base_channels)
        self.out = nn.Conv2d(base_channels, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)


binary_selector_model = None
use_binary_selector = BINARY_MODEL_PATH.exists()
if use_binary_selector:
    binary_checkpoint = torch.load(BINARY_MODEL_PATH, map_location=DEVICE)
    binary_base_channels = int(binary_checkpoint.get("base_channels", 16))
    binary_selector_model = SimpleUNet2D(out_channels=1, base_channels=binary_base_channels).to(DEVICE)
    binary_selector_model.load_state_dict(binary_checkpoint["model_state_dict"])
    binary_selector_model.eval()
    print("Selector binario cargado para center_window_best_prediction.")
else:
    print("ADVERTENCIA: no se encontro modelo binario. Se usara fallback central_slice.")

In [ ]:
def robust_percentile_normalize(image_array, p_low=1, p_high=99, eps=1e-8):
    image_float = image_array.astype(np.float32)
    low, high = np.percentile(image_float, [p_low, p_high])
    clipped = np.clip(image_float, low, high)
    if np.isclose(high, low):
        return np.zeros_like(clipped, dtype=np.float32)
    return ((clipped - low) / (high - low + eps)).astype(np.float32)


def take_slice(array, axis, index):
    return np.take(array, indices=index, axis=axis)


def resize_slice(array_2d, target_size=TARGET_SIZE, order=1):
    return resize(
        array_2d,
        output_shape=target_size,
        order=order,
        preserve_range=True,
        anti_aliasing=(order != 0),
    ).astype(np.float32)


def group_mask(mask_slice):
    grouped = np.zeros_like(mask_slice, dtype=np.int64)
    unknown_labels = sorted(set(np.unique(mask_slice).astype(int).tolist()) - set(original_to_group.keys()))
    if unknown_labels:
        print("ADVERTENCIA: labels no mapeados se enviaran a fondo:", unknown_labels)
    for original_label, group_id in original_to_group.items():
        grouped[mask_slice == original_label] = group_id
    return grouped.astype(np.int64)


def center_window_indices(volume_shape, radius=CENTER_WINDOW_RADIUS):
    center = int(volume_shape[SAGITTAL_AXIS] // 2)
    start = max(0, center - radius)
    end = min(volume_shape[SAGITTAL_AXIS] - 1, center + radius)
    return list(range(start, end + 1))


def select_slice_without_mask(image_norm):
    if binary_selector_model is None:
        return int(image_norm.shape[SAGITTAL_AXIS] // 2), "central_slice_fallback"

    indices = center_window_indices(image_norm.shape)
    image_slices = []
    for idx in indices:
        image_slice = take_slice(image_norm, SAGITTAL_AXIS, idx)
        image_slice = resize_slice(image_slice, order=1)
        image_slices.append(image_slice)

    probs = []
    with torch.no_grad():
        for start in range(0, len(image_slices), INFERENCE_BATCH_SIZE):
            batch_images = np.stack(image_slices[start:start + INFERENCE_BATCH_SIZE], axis=0)
            batch_tensor = torch.from_numpy(batch_images[:, None, :, :]).float().to(DEVICE)
            logits = binary_selector_model(batch_tensor)
            batch_probs = torch.sigmoid(logits).detach().cpu().numpy()[:, 0]
            probs.extend(list(batch_probs))

    foreground_scores = [float((prob >= 0.5).mean()) for prob in probs]
    best_local_idx = int(np.argmax(foreground_scores))
    return int(indices[best_local_idx]), "center_window_best_prediction"


def preprocess_case_multiclass(row):
    image_path, mask_path = get_case_paths(row)
    _, image = read_mha(image_path)
    _, mask = read_mha(mask_path)
    if image.shape != mask.shape:
        raise ValueError(f"Shape incompatible: image={image.shape}, mask={mask.shape}")

    image_norm = robust_percentile_normalize(image)
    slice_index, slice_strategy = select_slice_without_mask(image_norm)

    image_slice = resize_slice(take_slice(image_norm, SAGITTAL_AXIS, slice_index), order=1)
    mask_slice_original = take_slice(mask, SAGITTAL_AXIS, slice_index)
    mask_slice_original = resize_slice(mask_slice_original, order=0).astype(mask.dtype)
    mask_grouped = group_mask(mask_slice_original)

    unique_grouped = set(np.unique(mask_grouped).tolist())
    if not unique_grouped.issubset(set(range(NUM_CLASSES))):
        raise ValueError(f"Valores fuera de rango en mascara agrupada: {unique_grouped}")

    return {
        "case_id": str(row["case_id"]),
        "image": image_slice[None, ...].astype(np.float32),
        "mask": mask_grouped.astype(np.int64),
        "slice_index": int(slice_index),
        "slice_strategy": slice_strategy,
        "source_image_path": str(image_path),
        "source_mask_path": str(mask_path),
    }


preview = preprocess_case_multiclass(selected_df.iloc[0])
print("Preview image:", preview["image"].shape, preview["image"].dtype)
print("Preview mask:", preview["mask"].shape, preview["mask"].dtype, np.unique(preview["mask"]))
print("Slice strategy:", preview["slice_strategy"], "slice:", preview["slice_index"])

## 7. Dataset y split train/validation

In [ ]:
class SpiderMulticlassSagittalDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)
        self.cache = {}

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        if index not in self.cache:
            self.cache[index] = preprocess_case_multiclass(self.dataframe.iloc[index])
        item = self.cache[index]
        return {
            "image": torch.from_numpy(item["image"]).float(),
            "mask": torch.from_numpy(item["mask"]).long(),
            "case_id": item["case_id"],
            "slice_index": item["slice_index"],
            "slice_strategy": item["slice_strategy"],
            "source_image_path": item["source_image_path"],
            "source_mask_path": item["source_mask_path"],
        }


dataset = SpiderMulticlassSagittalDataset(selected_df)
val_size = max(1, int(round(0.2 * len(dataset))))
train_size = len(dataset) - val_size
generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)

split_rows = []
for split_name, subset in [("train", train_dataset), ("validation", val_dataset)]:
    for dataset_index in subset.indices:
        row = selected_df.iloc[dataset_index].to_dict()
        row["split"] = split_name
        split_rows.append(row)
split_df = pd.DataFrame(split_rows)
split_csv_path = MULTICLASS_ROOT / "E5_multiclass_split.csv"
split_df.to_csv(split_csv_path, index=False)

batch = next(iter(train_loader))
print("Train cases:", len(train_dataset))
print("Validation cases:", len(val_dataset))
print("Image tensor:", batch["image"].shape)
print("Mask tensor:", batch["mask"].shape, batch["mask"].dtype)
print("Mask values:", torch.unique(batch["mask"]))
print("CSV split:", split_csv_path)

## 8. Modelo multiclase

In [ ]:
multiclass_model = SimpleUNet2D(in_channels=1, out_channels=NUM_CLASSES, base_channels=BASE_CHANNELS).to(DEVICE)
logits_preview = multiclass_model(batch["image"].to(DEVICE))
print("Salida modelo:", logits_preview.shape)

## 9. Funcion de perdida y pesos de clase

In [ ]:
def estimate_class_pixel_counts(loader, num_classes=NUM_CLASSES):
    counts = np.zeros(num_classes, dtype=np.float64)
    for batch in tqdm(loader, desc="Estimando pesos de clase"):
        masks = batch["mask"].numpy()
        labels, label_counts = np.unique(masks, return_counts=True)
        for label, count in zip(labels, label_counts):
            counts[int(label)] += int(count)
    return counts


class_pixel_counts = estimate_class_pixel_counts(train_loader)
class_frequencies = class_pixel_counts / max(class_pixel_counts.sum(), 1.0)
raw_weights = 1.0 / np.clip(class_frequencies, 1e-6, None)
normalized_weights = raw_weights / raw_weights.mean()
clipped_weights = np.clip(normalized_weights, 0.25, 10.0).astype(np.float32)

class_weights = torch.tensor(clipped_weights, dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(multiclass_model.parameters(), lr=LEARNING_RATE)

class_weights_payload = {
    "class_pixel_counts": class_pixel_counts.astype(int).tolist(),
    "class_frequencies": class_frequencies.tolist(),
    "class_weights": clipped_weights.tolist(),
}
class_weights_json_path = MULTICLASS_ROOT / "E5_multiclass_class_weights.json"
class_weights_json_path.write_text(json.dumps(class_weights_payload, indent=2), encoding="utf-8")

print("Class counts:", class_pixel_counts)
print("Class frequencies:", class_frequencies)
print("Class weights:", clipped_weights)
print("Pesos exportados:", class_weights_json_path)

## 10. Metricas multiclase

In [ ]:
def multiclass_metrics_from_logits(logits, targets, num_classes=NUM_CLASSES, eps=1e-7):
    preds = torch.argmax(logits, dim=1)
    targets = targets.long()

    dice_by_class = []
    iou_by_class = []
    for class_id in range(num_classes):
        pred_c = preds == class_id
        target_c = targets == class_id
        intersection = torch.logical_and(pred_c, target_c).sum().float()
        pred_sum = pred_c.sum().float()
        target_sum = target_c.sum().float()
        union = torch.logical_or(pred_c, target_c).sum().float()
        dice = (2.0 * intersection + eps) / (pred_sum + target_sum + eps)
        iou = (intersection + eps) / (union + eps)
        dice_by_class.append(float(dice.detach().cpu()))
        iou_by_class.append(float(iou.detach().cpu()))

    dice_macro_no_bg = float(np.mean(dice_by_class[1:]))
    iou_macro_no_bg = float(np.mean(iou_by_class[1:]))
    pixel_accuracy = float((preds == targets).float().mean().detach().cpu())

    return {
        "dice_by_class": dice_by_class,
        "iou_by_class": iou_by_class,
        "dice_macro_no_bg": dice_macro_no_bg,
        "iou_macro_no_bg": iou_macro_no_bg,
        "pixel_accuracy": pixel_accuracy,
    }

## 11. Entrenamiento

In [ ]:
best_val_dice = -1.0
best_epoch = 0
best_model_path = MULTICLASS_ROOT / "E5_multiclass_unet2d_grouped_best.pt"
last_model_path = MULTICLASS_ROOT / "E5_multiclass_unet2d_grouped_last.pt"

def run_epoch(loader, train=False):
    multiclass_model.train() if train else multiclass_model.eval()
    losses = []
    dice_values = []
    iou_values = []
    acc_values = []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for batch in loader:
            images = batch["image"].to(DEVICE)
            masks = batch["mask"].to(DEVICE)
            logits = multiclass_model(images)
            loss = criterion(logits, masks)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            metrics = multiclass_metrics_from_logits(logits.detach(), masks)
            losses.append(float(loss.detach().cpu()))
            dice_values.append(metrics["dice_macro_no_bg"])
            iou_values.append(metrics["iou_macro_no_bg"])
            acc_values.append(metrics["pixel_accuracy"])
    return {
        "loss": float(np.mean(losses)),
        "dice_macro_no_bg": float(np.mean(dice_values)),
        "iou_macro_no_bg": float(np.mean(iou_values)),
        "pixel_accuracy": float(np.mean(acc_values)),
    }


history = []
for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, train=True)
    val_metrics = run_epoch(val_loader, train=False)
    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "val_loss": val_metrics["loss"],
        "val_dice_macro_no_bg": val_metrics["dice_macro_no_bg"],
        "val_iou_macro_no_bg": val_metrics["iou_macro_no_bg"],
        "val_pixel_accuracy": val_metrics["pixel_accuracy"],
    }
    history.append(row)
    print(row)

    if val_metrics["dice_macro_no_bg"] > best_val_dice:
        best_val_dice = val_metrics["dice_macro_no_bg"]
        best_epoch = epoch
        torch.save({
            "model_state_dict": multiclass_model.state_dict(),
            "epoch": epoch,
            "val_dice_macro_no_bg": best_val_dice,
            "num_classes": NUM_CLASSES,
            "label_group_mapping": LABEL_GROUP_MAPPING,
            "class_weights": clipped_weights.tolist(),
            "target_size": TARGET_SIZE,
            "sagittal_axis": SAGITTAL_AXIS,
            "base_channels": BASE_CHANNELS,
            "slice_strategy": "center_window_best_prediction" if binary_selector_model is not None else "central_slice_fallback",
        }, best_model_path)

torch.save({
    "model_state_dict": multiclass_model.state_dict(),
    "history": history,
    "num_classes": NUM_CLASSES,
    "label_group_mapping": LABEL_GROUP_MAPPING,
    "class_weights": clipped_weights.tolist(),
    "target_size": TARGET_SIZE,
    "sagittal_axis": SAGITTAL_AXIS,
    "base_channels": BASE_CHANNELS,
}, last_model_path)

history_df = pd.DataFrame(history)
history_csv_path = MULTICLASS_ROOT / "E5_multiclass_training_history.csv"
history_df.to_csv(history_csv_path, index=False)

print("Mejor epoch:", best_epoch)
print("Mejor modelo:", best_model_path)
print("Ultimo modelo:", last_model_path)
print("History:", history_csv_path)
display(history_df)

## 12. Evaluacion en validacion

In [ ]:
best_checkpoint = torch.load(best_model_path, map_location=DEVICE)
multiclass_model.load_state_dict(best_checkpoint["model_state_dict"])
multiclass_model.eval()

case_rows = []
class_metric_accumulator = {
    class_id: {"dice": [], "iou": []}
    for class_id in range(NUM_CLASSES)
}
visualization_items = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Evaluando validacion"):
        images = batch["image"].to(DEVICE)
        masks = batch["mask"].to(DEVICE)
        logits = multiclass_model(images)
        preds = torch.argmax(logits, dim=1)
        metrics = multiclass_metrics_from_logits(logits, masks)

        for class_id in range(NUM_CLASSES):
            class_metric_accumulator[class_id]["dice"].append(metrics["dice_by_class"][class_id])
            class_metric_accumulator[class_id]["iou"].append(metrics["iou_by_class"][class_id])

        case_rows.append({
            "case_id": batch["case_id"][0],
            "slice_index": int(batch["slice_index"][0]),
            "slice_strategy": batch["slice_strategy"][0],
            "dice_macro_no_bg": metrics["dice_macro_no_bg"],
            "iou_macro_no_bg": metrics["iou_macro_no_bg"],
            "pixel_accuracy": metrics["pixel_accuracy"],
            **{f"dice_class_{i}": metrics["dice_by_class"][i] for i in range(NUM_CLASSES)},
            **{f"iou_class_{i}": metrics["iou_by_class"][i] for i in range(NUM_CLASSES)},
        })

        if len(visualization_items) < 3:
            visualization_items.append({
                "case_id": batch["case_id"][0],
                "image": images.detach().cpu().numpy()[0, 0],
                "target": masks.detach().cpu().numpy()[0],
                "pred": preds.detach().cpu().numpy()[0],
            })

metrics_by_case_df = pd.DataFrame(case_rows)
metrics_by_case_csv_path = MULTICLASS_ROOT / "E5_multiclass_metrics_by_case.csv"
metrics_by_case_df.to_csv(metrics_by_case_csv_path, index=False)

class_rows = []
for class_id in range(NUM_CLASSES):
    class_rows.append({
        "class_id": class_id,
        "class_name": LABEL_GROUP_MAPPING[str(class_id)]["name"],
        "dice_mean": float(np.mean(class_metric_accumulator[class_id]["dice"])),
        "dice_std": float(np.std(class_metric_accumulator[class_id]["dice"])),
        "iou_mean": float(np.mean(class_metric_accumulator[class_id]["iou"])),
        "iou_std": float(np.std(class_metric_accumulator[class_id]["iou"])),
    })
metrics_by_class_df = pd.DataFrame(class_rows)
metrics_by_class_csv_path = MULTICLASS_ROOT / "E5_multiclass_metrics_by_class.csv"
metrics_by_class_df.to_csv(metrics_by_class_csv_path, index=False)

summary = {
    "val_dice_macro_no_bg": float(metrics_by_case_df["dice_macro_no_bg"].mean()),
    "val_iou_macro_no_bg": float(metrics_by_case_df["iou_macro_no_bg"].mean()),
    "val_pixel_accuracy": float(metrics_by_case_df["pixel_accuracy"].mean()),
    "best_epoch": int(best_epoch),
}
summary_df = pd.DataFrame([summary])
summary_csv_path = MULTICLASS_ROOT / "E5_multiclass_summary.csv"
summary_df.to_csv(summary_csv_path, index=False)

print("Metrics by case:", metrics_by_case_csv_path)
print("Metrics by class:", metrics_by_class_csv_path)
print("Summary:", summary_csv_path)
display(summary_df)
display(metrics_by_class_df)

## 13. Visualizaciones

In [ ]:
cmap = plt.get_cmap("tab10", NUM_CLASSES)

def export_multiclass_figure(item, index):
    error_map = (item["target"] != item["pred"]).astype(np.float32)
    fig, axes = plt.subplots(1, 6, figsize=(24, 4))

    axes[0].imshow(item["image"], cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("Imagen")

    axes[1].imshow(item["target"], cmap=cmap, vmin=0, vmax=NUM_CLASSES - 1)
    axes[1].set_title("GT agrupada")

    axes[2].imshow(item["pred"], cmap=cmap, vmin=0, vmax=NUM_CLASSES - 1)
    axes[2].set_title("Prediccion")

    axes[3].imshow(item["image"], cmap="gray", vmin=0, vmax=1)
    axes[3].imshow(np.ma.masked_where(item["target"] == 0, item["target"]), cmap=cmap, alpha=0.45, vmin=0, vmax=NUM_CLASSES - 1)
    axes[3].set_title("Overlay GT")

    axes[4].imshow(item["image"], cmap="gray", vmin=0, vmax=1)
    axes[4].imshow(np.ma.masked_where(item["pred"] == 0, item["pred"]), cmap=cmap, alpha=0.45, vmin=0, vmax=NUM_CLASSES - 1)
    axes[4].set_title("Overlay pred")

    axes[5].imshow(error_map, cmap="Reds", vmin=0, vmax=1)
    axes[5].set_title("Mapa de error")

    for ax in axes:
        ax.axis("off")

    fig.suptitle(f"Multiclase agrupado - {item['case_id']}")
    fig.tight_layout()
    path = FIGURES_ROOT / f"E5_multiclass_prediction_example_{index:02d}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    return path


prediction_figure_paths = []
for index, item in enumerate(visualization_items, start=1):
    prediction_figure_paths.append(export_multiclass_figure(item, index))

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train_loss")
ax.plot(history_df["epoch"], history_df["val_loss"], marker="o", label="val_loss")
ax.set_title("E5 multiclase - Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
loss_curve_png_path = FIGURES_ROOT / "E5_multiclass_loss_curve.png"
fig.savefig(loss_curve_png_path, dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(history_df["epoch"], history_df["val_dice_macro_no_bg"], marker="o", label="val_dice_macro_no_bg")
ax.set_title("E5 multiclase - Dice macro sin fondo")
ax.set_xlabel("Epoch")
ax.set_ylabel("Dice")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
dice_curve_png_path = FIGURES_ROOT / "E5_multiclass_dice_curve.png"
fig.savefig(dice_curve_png_path, dpi=150, bbox_inches="tight")
plt.show()

print("Figuras:", prediction_figure_paths)
print("Loss curve:", loss_curve_png_path)
print("Dice curve:", dice_curve_png_path)

## 14. Comparacion conceptual contra binario

In [ ]:
binary_context = {
    "binary_holdout_dice": 0.8816,
    "binary_holdout_iou": 0.7904,
    "multiclass_val_dice_macro_no_bg": summary["val_dice_macro_no_bg"],
    "multiclass_val_iou_macro_no_bg": summary["val_iou_macro_no_bg"],
    "note": "No son metricas directamente equivalentes: binario mide foreground/background y multiclase mide separacion entre grupos tecnicos.",
}
binary_context_df = pd.DataFrame([binary_context])
binary_context_csv_path = MULTICLASS_ROOT / "E5_multiclass_vs_binary_context.csv"
binary_context_df.to_csv(binary_context_csv_path, index=False)

display(binary_context_df)
print("CSV contexto binario:", binary_context_csv_path)

## 15. Reporte JSON

In [ ]:
report = {
    "n_cases": int(len(selected_df)),
    "train_cases": int(len(train_dataset)),
    "validation_cases": int(len(val_dataset)),
    "modality": MODALITY_FILTER,
    "num_classes": NUM_CLASSES,
    "mapping": LABEL_GROUP_MAPPING,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "device": str(DEVICE),
    "class_weights": clipped_weights.tolist(),
    "best_epoch": int(best_epoch),
    "val_dice_macro_no_bg": summary["val_dice_macro_no_bg"],
    "val_iou_macro_no_bg": summary["val_iou_macro_no_bg"],
    "val_pixel_accuracy": summary["val_pixel_accuracy"],
    "dice_by_class": {
        str(row["class_id"]): float(row["dice_mean"])
        for _, row in metrics_by_class_df.iterrows()
    },
    "iou_by_class": {
        str(row["class_id"]): float(row["iou_mean"])
        for _, row in metrics_by_class_df.iterrows()
    },
    "exports": {
        "selected_cases": str(selected_cases_csv_path),
        "original_label_distribution": str(original_label_distribution_csv_path),
        "label_mapping": str(mapping_json_path),
        "split": str(split_csv_path),
        "class_weights": str(class_weights_json_path),
        "history": str(history_csv_path),
        "metrics_by_case": str(metrics_by_case_csv_path),
        "metrics_by_class": str(metrics_by_class_csv_path),
        "summary": str(summary_csv_path),
        "best_model": str(best_model_path),
        "last_model": str(last_model_path),
        "prediction_figures": [str(path) for path in prediction_figure_paths],
        "loss_curve": str(loss_curve_png_path),
        "dice_curve": str(dice_curve_png_path),
        "binary_context": str(binary_context_csv_path),
    },
}

report_json_path = MULTICLASS_ROOT / "E5_multiclass_validation_report.json"
report_json_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("Reporte JSON:", report_json_path)
print(json.dumps(report, indent=2))

## 16. Conclusion tecnica Markdown

In [ ]:
conclusion_md = f"""# Conclusion tecnica - E5 Baseline multiclase agrupado sagital

## Objetivo

Se construyo un primer baseline multiclase 2D sagital sobre SPIDER para evaluar si el modelo puede diferenciar grupos tecnicos de labels, partiendo del pipeline binario ya validado.

## Por que pasar de binario a multiclase agrupado

El baseline binario valido el flujo de segmentacion foreground/background. La formulacion multiclase agrupada agrega dificultad al exigir separacion entre grupos de etiquetas, pero evita todavia asumir nombres anatomicos definitivos o pasar directamente a una segmentacion completa estructura por estructura.

## Configuracion

- Dataset: SPIDER
- Modalidad: `{MODALITY_FILTER}`
- Casos seleccionados: {len(selected_df)}
- Train cases: {len(train_dataset)}
- Validation cases: {len(val_dataset)}
- Eje sagital: {SAGITTAL_AXIS}
- Target size: {TARGET_SIZE}
- NUM_CLASSES: {NUM_CLASSES}
- Device: `{DEVICE}`
- Epochs: {EPOCHS}
- Batch size: {BATCH_SIZE}
- Selector de slice: {'center_window_best_prediction' if binary_selector_model is not None else 'central_slice_fallback'}

## Mapping de labels usado

```json
{json.dumps(LABEL_GROUP_MAPPING, indent=2)}
```

Este mapping es tecnico y preliminar. No asigna nombres anatomicos definitivos porque debe validarse contra documentacion formal del dataset.

## Resultados principales

- Mejor epoch: {best_epoch}
- Dice macro sin fondo: {summary['val_dice_macro_no_bg']:.4f}
- IoU macro sin fondo: {summary['val_iou_macro_no_bg']:.4f}
- Pixel accuracy secundaria: {summary['val_pixel_accuracy']:.4f}

## Comportamiento por clase

{metrics_by_class_df.to_markdown(index=False)}

## Comparacion conceptual contra binario

El baseline binario holdout obtuvo Dice 0.8816 e IoU 0.7904. Esos valores no son directamente comparables con las metricas multiclase porque el problema binario mide foreground/background, mientras que el multiclase agrupado mide separacion entre grupos tecnicos y es una tarea mas dificil.

## Evidencias exportadas

- Distribucion labels originales: `{original_label_distribution_csv_path}`
- Mapping labels: `{mapping_json_path}`
- Split: `{split_csv_path}`
- Pesos de clase: `{class_weights_json_path}`
- History: `{history_csv_path}`
- Metricas por caso: `{metrics_by_case_csv_path}`
- Metricas por clase: `{metrics_by_class_csv_path}`
- Summary: `{summary_csv_path}`
- Mejor modelo: `{best_model_path}`
- Reporte JSON: `{report_json_path}`

## Limitaciones

- Mapping agrupado preliminar.
- No se asignan nombres anatomicos definitivos sin documentacion formal.
- Entrenamiento 2D sobre un slice.
- No hay inferencia 3D.
- No se evalua axial.
- No se usa nnU-Net.
- Puede existir desbalance entre clases.
- Validacion interna, no clinica.

## Recomendacion para proximo paso

Si el Dice macro sin fondo y las metricas por clase muestran aprendizaje razonable, conviene profundizar con sampling balanceado por clase, augmentations y una evaluacion por estructura mas detallada. Si una o mas clases tienen metricas bajas, antes de avanzar a nnU-Net conviene revisar mapping, frecuencia de labels y seleccion de slices.
"""

conclusion_path = DOCS_ROOT / "E5_multiclase_agrupado_conclusion.md"
conclusion_path.write_text(conclusion_md, encoding="utf-8")

print(conclusion_md)
print("Conclusion Markdown:", conclusion_path)

## 17. Criterio de aceptacion

El notebook es correcto si analiza labels originales, define y exporta un mapping agrupado, entrena una U-Net 2D multiclase, calcula Dice/IoU macro excluyendo fondo, calcula metricas por clase, exporta CSVs, figuras, modelo, JSON y Markdown, y deja claro si el enfoque multiclase agrupado es viable o si conviene mejorar dataset/sampling antes de avanzar.

No usar nnU-Net, axial ni backend en este notebook.